# 02 (Qwen3-8B) Residual Stream / Logit Lens / Activation Patching

これは `02_residual_stream_logit_lens_patching.ipynb`（Qwen3-4B 版）の派生ノートです。
同じ実験を **Qwen3-8B** で実行し、4B との挙動の差を確認します。

読者は 4B 版を既読として、各手法の概念説明・hook の詳細・概念図などは省略します。
代わりに **アーキテクチャとトークナイザーの 4B との差分** と、**結果の 4B との比較** を中心に扱います。

実験設定は 4B 版と同じです：

| | プロンプト | 期待する次トークン |
|---|---|---|
| **clean** | `"The capital of Japan is"` | ` Tokyo` |
| **corrupt** | `"The capital of France is"` | ` Paris` |


## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [ ]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

## 1. 環境セットアップ

In [ ]:
%matplotlib inline
import math
import json
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

MODEL_ID = "Qwen/Qwen3-8B"  # 4B 版との唯一の主要な差

outputs_dir = (Path("outputs") if IN_COLAB else Path("../outputs"))
outputs_dir.mkdir(parents=True, exist_ok=True)
print(f"model_id : {MODEL_ID}")
print(f"outputs  : {outputs_dir.name}/")

# デバイス選択（4B 版と同じ）
if torch.cuda.is_available():
    device = "cuda"
    torch_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
print(f"device   : {device}")
print(f"dtype    : {torch_dtype}")


## 2. モデルとトークナイザーの読み込み

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"  vocab_size = {tokenizer.vocab_size}")

print("Loading model ...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    attn_implementation="eager",
)
model.to(device)  # pyright: ignore[reportArgumentType]
model.eval()
torch.set_grad_enabled(False)
print(f"  device     : {next(model.parameters()).device}")
print(f"  dtype      : {next(model.parameters()).dtype}")

K: int = model.config.num_hidden_layers
hidden_size: int = model.config.hidden_size
print(f"  layers K   : {K}")
print(f"  hidden_size: {hidden_size}")


## 3. Qwen3-4B との差分確認

このノートは 4B 版のデモを 8B で動かすものなので、まず両モデルの
**アーキテクチャ** と **トークナイザー** がどこで違うのか（あるいは同じか）を明示しておく。


### 3.1 アーキテクチャ差分

In [ ]:
# config の主要フィールドを 4B の既知値と並べて確認する。
# 4B の値はノート 02（4B 版）で実測した値を参考にハードコードする
# （このノート内では 8B しか読み込んでいないため、両者を読み込み直すのは避ける）。
cfg = model.config

# 4B 版で確認済みの値（CLAUDE.md にも記載されている shape 情報と整合）
ref_4b = {
    "num_hidden_layers":     36,
    "hidden_size":           2560,
    "num_attention_heads":   32,
    "num_key_value_heads":   8,
    "head_dim":              128,
    "intermediate_size":     9728,
    "vocab_size":            151936,
    "tie_word_embeddings":   True,
    "max_position_embeddings": 40960,
}

fields = list(ref_4b.keys())
rows = []
for f in fields:
    v_this = getattr(cfg, f, None)
    v_4b = ref_4b[f]
    rows.append({
        "field":     f,
        "Qwen3-8B": v_this,
        "Qwen3-4B (参考)": v_4b,
        "同一":       v_this == v_4b,
    })

df_arch = pd.DataFrame(rows).set_index("field")
display(df_arch.style.set_caption("Qwen3-8B vs Qwen3-4B  config 差分"))


**観察結果**：

- **`num_hidden_layers` (K)**: 8B は **36**（4B と**完全に同じ**深さ）。
  「8B は 4B より大きい」と聞くと深さも増えそうに思えるが、Qwen3 系列ではこの 2 モデルで
  layer 数は同じ。サイズの差は主に「幅」で実現されている。
- **`hidden_size`**: 8B は **4096**（4B の 2560 の **1.6 倍**広い）。
- **`num_attention_heads`**: 8B は **32**（4B と**同じ**）。
- **`num_key_value_heads`**: 両者とも **8**（GQA 構造・共有度は共通）。
  Q ヘッド / KV ヘッド比は 32/8 = 4 で 4B と同じ。
- **`head_dim`**: 両者とも **128**。head 数 × head_dim = 32 × 128 = 4096 = `hidden_size`（8B）、
  4B では 32 × 128 = 4096 ではなく **2560**（つまり 4B は head の合計次元 < hidden_size、
  Q/K/V の射影で次元圧縮あり）。8B では一致するので、Q/K/V 射影は次元保存型。
- **`intermediate_size`**: 8B は **12288**（4B の 9728 の 1.26 倍）。
- **`vocab_size` (151936)**, **`max_position_embeddings` (40960)**: 両者で一致。
- **`tie_word_embeddings`**: 8B は **`False`**（4B の `True` と**異なる**）。
  → 8B では embedding と lm_head の重みが**独立**して学習される（`W_E ≠ W_U`）。
    予想どおり、Qwen3 系列はサイズが大きくなると embedding を共有しなくなる。
    logit lens の計算自体は影響を受けない（`lm_head` を通すだけなので）が、
    embedding 空間と output 空間の関係が 4B と質的に異なる点に注意。

要約すると、8B は **layer 数・head 数・head_dim・vocab・最大文脈長は 4B と同じ**で、
hidden_size と intermediate_size を広げ、さらに embedding を untie することでスケールアップしている。


### 3.2 トークナイザー差分

In [ ]:
# 4B と 8B は同じ Qwen3 系列なので、トークナイザーも同一であることを期待する。
# clean/corrupt の答え（" Tokyo" / " Paris"）が同じ token ID にマップされるかを
# 改めて確認しておく（これが崩れると 4B との直接比較が難しくなる）。

CLEAN_ANSWER   = " Tokyo"
CORRUPT_ANSWER = " Paris"

clean_ans_ids   = tokenizer.encode(CLEAN_ANSWER,   add_special_tokens=False)
corrupt_ans_ids = tokenizer.encode(CORRUPT_ANSWER, add_special_tokens=False)

# 4B で確認した token ID（02 ノート実行時の値を参考にハードコード）
ref_4b_ids = {
    " Tokyo": 26194,
    " Paris": 12095,
}

rows = []
for ans, ids in [(CLEAN_ANSWER, clean_ans_ids), (CORRUPT_ANSWER, corrupt_ans_ids)]:
    rows.append({
        "answer":         repr(ans),
        "ids (8B)":     ids,
        "id (4B, 参考)":  ref_4b_ids[ans],
        "single token":   len(ids) == 1,
        "同一":           len(ids) == 1 and ids[0] == ref_4b_ids[ans],
    })
display(pd.DataFrame(rows).style.set_caption("answer token の 4B との一致確認"))

CLEAN_ANS_ID   = clean_ans_ids[0]
CORRUPT_ANS_ID = corrupt_ans_ids[0]
print(f"CLEAN_ANS_ID   = {CLEAN_ANS_ID}   ({CLEAN_ANSWER!r})")
print(f"CORRUPT_ANS_ID = {CORRUPT_ANS_ID}  ({CORRUPT_ANSWER!r})")

# vocab_size の整合
print(f"tokenizer.vocab_size      = {tokenizer.vocab_size}")
print(f"model.config.vocab_size   = {model.config.vocab_size}")


**観察結果**：

- `" Tokyo"` → **26194**、`" Paris"` → **12095**。両者とも 4B と完全一致した（**同一 = True**）。
- `tokenizer.vocab_size = 151643` と `model.config.vocab_size = 151936` の間に 293 のずれがあるが、
  これは tokenizer 本体の語彙数（base vocab）と、それに追加された special token を含む
  拡張後 vocab size の差（4B でも同じ値）。logit lens / patching では `model.config.vocab_size`
  側（lm_head の出力次元）が使われるので問題ない。
- 結論：**Qwen3-8B と Qwen3-4B はトークナイザー完全一致**。同じ prompt は同じ token 列に
  分解され、同じ answer token ID で確率を比較できる。8B と 4B（さらに 1.7B）の `recovery`
  や P(answer) の数値は **3 モデルすべて直接比較可能**。


## 4. プロンプトとトークンテーブル

In [ ]:
CLEAN_PROMPT   = "The capital of Japan is"
CORRUPT_PROMPT = "The capital of France is"

def show_token_table(text: str) -> pd.DataFrame:
    ids = tokenizer.encode(text, add_special_tokens=False)
    rows = []
    for pos, tid in enumerate(ids):
        piece = tokenizer.convert_ids_to_tokens(tid)
        decoded = tokenizer.decode([tid])
        rows.append({
            "pos":      pos,
            "token_id": tid,
            "piece":    piece,
            "decoded":  repr(decoded),
        })
    return pd.DataFrame(rows).set_index("pos")

def topk_table(logits: torch.Tensor, k: int = 10) -> pd.DataFrame:
    probs = torch.softmax(logits.float(), dim=-1)
    top_vals, top_ids = torch.topk(probs, k)
    rows = []
    for rank, (tid, prob) in enumerate(zip(top_ids.tolist(), top_vals.tolist()), start=1):
        decoded = tokenizer.decode([tid])
        rows.append({
            "rank":     rank,
            "token_id": tid,
            "decoded":  repr(decoded),
            "logit":    logits[tid].item(),
            "prob":     prob,
        })
    return pd.DataFrame(rows).set_index("rank")

df_clean_tok = show_token_table(CLEAN_PROMPT)
display(df_clean_tok.style.set_caption("clean prompt token table"))
clean_pos = len(df_clean_tok) - 1

df_corrupt_tok = show_token_table(CORRUPT_PROMPT)
display(df_corrupt_tok.style.set_caption("corrupt prompt token table"))
corrupt_pos = len(df_corrupt_tok) - 1


## 5. clean / corrupt run の next-token 分布

In [ ]:
# clean run
clean_inputs = tokenizer(CLEAN_PROMPT, return_tensors="pt").to(device)
clean_outputs = model(
    **clean_inputs,
    output_hidden_states=True,
    output_attentions=False,
    use_cache=False,
)
clean_hs     = clean_outputs.hidden_states
clean_logits = clean_outputs.logits[0, clean_pos, :].float()
clean_probs  = torch.softmax(clean_logits, dim=-1)

# corrupt run
corrupt_inputs = tokenizer(CORRUPT_PROMPT, return_tensors="pt").to(device)
corrupt_outputs = model(
    **corrupt_inputs,
    output_hidden_states=True,
    output_attentions=False,
    use_cache=False,
)
corrupt_hs     = corrupt_outputs.hidden_states
corrupt_logits = corrupt_outputs.logits[0, corrupt_pos, :].float()
corrupt_probs  = torch.softmax(corrupt_logits, dim=-1)

print(f"clean   top-1 = {repr(tokenizer.decode([clean_logits.argmax().item()]))}"
      f"  P({CLEAN_ANSWER.strip()})={clean_probs[CLEAN_ANS_ID]:.4f}"
      f"  P({CORRUPT_ANSWER.strip()})={clean_probs[CORRUPT_ANS_ID]:.4f}")
print(f"corrupt top-1 = {repr(tokenizer.decode([corrupt_logits.argmax().item()]))}"
      f"  P({CLEAN_ANSWER.strip()})={corrupt_probs[CLEAN_ANS_ID]:.4f}"
      f"  P({CORRUPT_ANSWER.strip()})={corrupt_probs[CORRUPT_ANS_ID]:.4f}")

display(topk_table(clean_logits,   k=10).style.set_caption("clean top-10"))
display(topk_table(corrupt_logits, k=10).style.set_caption("corrupt top-10"))


**観察結果**：

| run | top-1 | P(` Tokyo`) | P(` Paris`) |
|-----|-------|-------------|-------------|
| clean (` Japan`) | ` Tokyo` ✓ | **0.4687** | 0.0001 |
| corrupt (` France`) | ` Paris` ✓ | 0.0003 | **0.5345** |

**4B との比較（および 1.7B を含めた 3 モデル）**：

| run | 4B | 1.7B | **8B** |
|---|---|---|---|
| clean P(` Tokyo`)  | 0.8946 | 0.4169 | **0.4687** |
| corrupt P(` Paris`) | 0.6346 | 0.5246 | **0.5345** |

- 8B でも top-1 は正しく ` Tokyo` / ` Paris` に立つ。
- **意外なこと**：8B の P(` Tokyo`) = 0.47 は 4B の 0.89 より **大幅に低い**。
  「大きいモデル＝確信度が上がる」という素朴な期待に反する結果。1.7B（0.42）とほぼ同じ水準で、
  むしろ「4B だけが突出して鋭い分布」というのが実状。
  考えられる理由：
  - 4B では `tie_word_embeddings = True` で embedding 空間と出力空間が結びついているため、
    具体的な単語に強い確率質量が集中しやすい
  - 8B は untied lm_head のため、output 分布がより分散しやすい（複数の妥当な候補へ確率が割れる）
  - 実際、8B の clean top-10 には ` Beijing`（rank 9, P=0.0131）が含まれており、
    アジア圏の首都が「妥当な代替候補」として捉えられているらしい
- top-10 のフィラー構成（` a` / ` __` / ` ______` / ` located` 等）は 4B / 1.7B と似ており、
  この prompt に対する分布の質的傾向は共通している。


## 6. Logit Lens

`logit_lens(hs, k, pos)` の実装は 4B 版と同一。


In [ ]:
def logit_lens(hidden_states, k: int, pos: int) -> torch.Tensor:
    K_loc = len(hidden_states) - 1
    hs = hidden_states[k]
    if k < K_loc:
        normed = model.model.norm(hs[:, pos:pos+1, :])
        logits = model.lm_head(normed)[:, 0, :]
    else:
        normed = hs[:, pos, :]
        logits = model.lm_head(normed)
    return logits[0]

def logit_lens_sweep(hidden_states, pos: int) -> pd.DataFrame:
    K_loc = len(hidden_states) - 1
    rows = []
    for k in range(K_loc + 1):
        ll_logits    = logit_lens(hidden_states, k, pos)
        ll_probs     = torch.softmax(ll_logits.float(), dim=-1)
        top1_id      = int(ll_logits.argmax().item())
        top1_decoded = tokenizer.decode([top1_id])
        site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K_loc else "norm")
        rows.append({
            "k":            k,
            "site":         site,
            "top1_decoded": repr(top1_decoded),
            "p_top1":       ll_probs[top1_id].item(),
            "p_clean":      ll_probs[CLEAN_ANS_ID].item(),
            "p_corrupt":    ll_probs[CORRUPT_ANS_ID].item(),
        })
    return pd.DataFrame(rows)

# sanity check: logit_lens(k=K) は最終 logits と一致するはず
ll_K = logit_lens(clean_hs, K, clean_pos)
diff_K = (ll_K - clean_logits).abs().max().item()
print(f"logit_lens(k=K) sanity diff = {diff_K:.6f}  (~0 なら OK)")


### 6.1 最後の position の logit lens — clean / corrupt

In [ ]:
df_ll_clean   = logit_lens_sweep(clean_hs,   clean_pos)
df_ll_corrupt = logit_lens_sweep(corrupt_hs, corrupt_pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax = axes[0]
ax.plot(df_ll_clean["k"], df_ll_clean["p_clean"],
        label=f"P({CLEAN_ANSWER})",   marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_corrupt"],
        label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — clean ({CLEAN_PROMPT})")
ax.set_xlabel("layer k")
ax.set_ylabel("probability")
ax.legend()
ax.set_xticks(range(0, K + 1, max(1, K // 9)))
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_clean"],
        label=f"P({CLEAN_ANSWER})",   marker="o", markersize=3)
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_corrupt"],
        label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — corrupt ({CORRUPT_PROMPT})")
ax.set_xlabel("layer k")
ax.legend()
ax.set_xticks(range(0, K + 1, max(1, K // 9)))
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = outputs_dir / "nb02_qwen3_8b_logit_lens_comparison.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


### 6.2 logit-difference metric — clean / corrupt

In [ ]:
def metric(logits: torch.Tensor, clean_id: int, corrupt_id: int) -> float:
    return (logits[clean_id] - logits[corrupt_id]).item()

ll_metric_clean   = [metric(logit_lens(clean_hs,   k, clean_pos),
                            CLEAN_ANS_ID, CORRUPT_ANS_ID) for k in range(K + 1)]
ll_metric_corrupt = [metric(logit_lens(corrupt_hs, k, corrupt_pos),
                            CLEAN_ANS_ID, CORRUPT_ANS_ID) for k in range(K + 1)]

ks = list(range(K + 1))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks, ll_metric_clean,   label=f"clean  ({CLEAN_PROMPT})",   marker="o", markersize=3)
ax.plot(ks, ll_metric_corrupt, label=f"corrupt ({CORRUPT_PROMPT})", marker="s", markersize=3, linestyle="--")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel(f"logit({CLEAN_ANSWER}) - logit({CORRUPT_ANSWER})")
ax.set_title("Logit Lens — metric_k  (logit difference) [Qwen3-8B]")
ax.legend()
ax.set_xticks(range(0, K + 1, max(1, K // 9)))
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = outputs_dir / "nb02_qwen3_8b_logit_lens_metric.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


### 6.3 Logit Lens Grid — clean / corrupt

In [ ]:
# 4B 版と同じ helper（短い token piece 表示 + grid 描画）
import matplotlib.font_manager as _fm
_available = {f.name for f in _fm.fontManager.ttflist}
_cjk_candidates = [
    # 日本語
    "Hiragino Sans", "Yu Gothic", "Meiryo", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
    # 簡体字/繁体字（Hiragino/Yu Gothic では簡体字が欠ける。Mac=PingFang SC / Win=Microsoft YaHei / Colab=WenQuanYi）
    "PingFang SC", "Microsoft YaHei", "WenQuanYi Zen Hei", "Noto Sans CJK SC",
    # アラビア語（RTL の並べ替えは _fix_rtl。字形はフォントから供給。Mac=Geeza Pro / Win=Segoe UI / Colab=Noto Arabic）
    "Geeza Pro", "Segoe UI", "Noto Sans Arabic", "Noto Naskh Arabic",
    "Arial Unicode MS",
]
_cjk_fonts = [name for name in _cjk_candidates if name in _available]
plt.rcParams["font.family"] = _cjk_fonts + ["DejaVu Sans"]

def short_piece(token_id: int, max_len: int = 10) -> str:
    s = tokenizer.decode([int(token_id)])
    s = s.replace("\n", "\\n").replace("\t", "\\t")
    s = "".join(c if ord(c) < 0x10000 else "□" for c in s)
    if s == "":
        s = "∅"
    if len(s) > max_len:
        s = s[:max_len] + "…"
    return _fix_rtl(s)

def plot_logit_lens_grid(hidden_states, input_ids, target_ids, title, out_path=None):
    K_loc = len(hidden_states) - 1
    T = len(input_ids)
    vocab_size = model.config.vocab_size
    log_M = math.log10(vocab_size)

    grid_text, rank_score, top1_match = [], [], []
    for k in range(K_loc + 1):
        row_t, row_s, row_m = [], [], []
        for pos in range(T):
            logits = logit_lens(hidden_states, k, pos).float().detach().cpu()
            target_id = int(target_ids[pos])
            target_logit = logits[target_id].item()
            rank = int((logits > target_logit).sum().item()) + 1
            top1_id = int(logits.argmax().item())
            score = 1.0 - math.log10(rank) / log_M
            row_t.append(short_piece(top1_id))
            row_s.append(score)
            row_m.append(top1_id == target_id)
        grid_text.append(row_t)
        rank_score.append(row_s)
        top1_match.append(row_m)

    score_arr = np.array(rank_score)
    match_arr = np.array(top1_match)
    fig_w = max(8.0, 1.6 * T)
    fig_h = max(10.0, 0.28 * (K_loc + 1))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    cmap = plt.get_cmap("viridis")
    im = ax.imshow(score_arr, aspect="auto", vmin=0.0, vmax=1.0, cmap=cmap, origin="lower")
    ax.set_title(title)
    ax.set_xlabel("Position: input token → target next token")
    ax.set_ylabel("Layer k")

    xlabels = [f"{short_piece(int(input_ids[t]))}→{short_piece(int(target_ids[t]))}"
               for t in range(T)]
    ax.set_xticks(range(T))
    ax.set_xticklabels(xlabels, rotation=45, ha="right")

    ylabels = []
    for k in range(K_loc + 1):
        if k == 0:
            ylabels.append("0 emb")
        elif k == K_loc:
            ylabels.append(f"{k} norm")
        else:
            ylabels.append(str(k))
    ax.set_yticks(range(K_loc + 1))
    ax.set_yticklabels(ylabels)

    for k in range(K_loc + 1):
        for pos in range(T):
            rgba = cmap(score_arr[k, pos])
            lum = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
            text_color = "white" if lum < 0.5 else "black"
            ax.text(pos, k, grid_text[k][pos],
                    ha="center", va="center", fontsize=7, color=text_color)
            if match_arr[k, pos]:
                ax.add_patch(Rectangle((pos - 0.5, k - 0.5), 1, 1,
                                       fill=False, edgecolor="red", linewidth=1.8))

    cbar = fig.colorbar(im, ax=ax)
    rank_ticks = [1, 10, 100, 1000, 10000, vocab_size]
    score_ticks = [1.0 - math.log10(r) / log_M for r in rank_ticks]
    cbar.set_ticks(score_ticks)
    cbar.set_ticklabels([f"{r:,}" for r in rank_ticks])
    cbar.set_label("target rank (log scale)")
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=200)
        print(f"saved: {out_path.name}")
    return fig

# === Colab: 多言語フォント（日本語/中国語/アラビア語/ベトナム語）===
# 方針: 各フォントを明示 addfont（IPAGothic単体で効いた方式）。キャッシュ削除/再構築はしない（日本語が壊れたため）。
import sys as _sysc
if "google.colab" in _sysc.modules:
    import subprocess as _spc, glob as _gc
    _spc.run(["apt-get", "-qq", "-y", "install",
              "fonts-ipafont-gothic", "fonts-wqy-zenhei", "fonts-noto-core"], check=False)
    _spc.run([_sysc.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=False)
    import matplotlib.font_manager as _fmj, matplotlib.pyplot as _pltj
    for _pat in ("/usr/share/fonts/**/ipag*.ttf",
                 "/usr/share/fonts/**/wqy-zenhei*.tt?",
                 "/usr/share/fonts/**/NotoSansArabic*.ttf",
                 "/usr/share/fonts/**/NotoNaskhArabic*.ttf",
                 "/usr/share/fonts/**/NotoSans-*.ttf"):
        for _fp in _gc.glob(_pat, recursive=True):
            try:
                _fmj.fontManager.addfont(_fp)
            except Exception:
                pass
    # font.family にリストを直接指定 → 字ごとに先頭から fallback
    _pltj.rcParams["font.family"] = ["IPAGothic", "WenQuanYi Zen Hei", "Noto Sans Arabic", "Noto Naskh Arabic", "Noto Sans", "DejaVu Sans"]
    _pltj.rcParams["axes.unicode_minus"] = False
    _reg = {f.name for f in _fmj.fontManager.ttflist}
    print("[Colab] 登録確認 IPAGothic=%s WenQuanYi=%s NotoArabic=%s" % ("IPAGothic" in _reg, "WenQuanYi Zen Hei" in _reg, "Noto Sans Arabic" in _reg))

# RTL(アラビア語/ヘブライ語)整形 — 環境依存に対応（自動判定＋手動上書き可）
# matplotlib が libraqm 付きでビルドされた環境(Mac の pip wheel 等)は Arabic を native 整形(連結＋右→左)するので
# raw が正しい。libraqm 非搭載の環境(古い/一部 Linux・Colab)だけ arabic-reshaper+python-bidi で手動整形が要る。
# これは matplotlib の版番号ではなく「バイナリ(libraqm リンク)」で決まるため、自動判定を既定にしつつ手動でも切替可能にする。
RESHAPE_RTL = "auto"   # "auto" = libraqm の有無で自動 / True = 必ず手動整形 / False = 整形しない(raw)
try:
    import matplotlib.ft2font as _ft2
    _MPL_HAS_RAQM = bool(getattr(_ft2, "__libraqm_version__", ""))
except Exception:
    _MPL_HAS_RAQM = False
def _need_manual_reshape():
    return (not _MPL_HAS_RAQM) if RESHAPE_RTL == "auto" else bool(RESHAPE_RTL)
try:
    import arabic_reshaper as _arsh
    from bidi.algorithm import get_display as _bidi_disp
    import re as _re_rtl
    _RTL_RE = _re_rtl.compile(r'[֐-ࣿﭐ-﷿ﹰ-﻿]')
    def _fix_rtl(_s):
        if _need_manual_reshape() and _s and _RTL_RE.search(_s):
            return _bidi_disp(_arsh.reshape(_s))
        return _s
except Exception:
    def _fix_rtl(_s):
        return _s
print(f"[RTL] libraqm={_MPL_HAS_RAQM}  RESHAPE_RTL={RESHAPE_RTL}  -> 手動整形={_need_manual_reshape()}")


In [ ]:
# clean grid
input_ids_clean = clean_inputs["input_ids"][0].detach().cpu().tolist()
target_ids_clean = input_ids_clean[1:] + [CLEAN_ANS_ID]
plot_logit_lens_grid(
    clean_hs, input_ids_clean, target_ids_clean,
    title=f"Logit lens grid: {CLEAN_PROMPT!r}  [Qwen3-8B]",
    out_path=outputs_dir / "nb02_qwen3_8b_logit_lens_grid_clean.png",
)
plt.show()


In [ ]:
# corrupt grid
input_ids_corrupt = corrupt_inputs["input_ids"][0].detach().cpu().tolist()
target_ids_corrupt = input_ids_corrupt[1:] + [CORRUPT_ANS_ID]
plot_logit_lens_grid(
    corrupt_hs, input_ids_corrupt, target_ids_corrupt,
    title=f"Logit lens grid: {CORRUPT_PROMPT!r}  [Qwen3-8B]",
    out_path=outputs_dir / "nb02_qwen3_8b_logit_lens_grid_corrupt.png",
)
plt.show()


## 7. Activation Patching — 最終 position

`run_patch(k)` の実装は 4B 版と同一。


In [ ]:
def run_patch(k: int) -> torch.Tensor:
    patch_vec = clean_hs[k][0, clean_pos, :].to(device)
    if k == 0:
        target_module = model.model.embed_tokens
    elif k < K:
        target_module = model.model.layers[k - 1]
    else:
        target_module = model.model.norm

    def hook(module, inp, out):
        out = out.clone()
        out[0, corrupt_pos, :] = patch_vec
        return out

    handle = target_module.register_forward_hook(hook)
    try:
        patched_out = model(
            **corrupt_inputs,
            output_hidden_states=False,
            output_attentions=False,
            use_cache=False,
        )
    finally:
        handle.remove()
    return patched_out.logits[0, corrupt_pos, :].float()

clean_metric   = metric(clean_logits,   CLEAN_ANS_ID, CORRUPT_ANS_ID)
corrupt_metric = metric(corrupt_logits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
print(f"clean   metric = {clean_metric:.4f}")
print(f"corrupt metric = {corrupt_metric:.4f}")

sweep_rows = []
print(f"Patching sweep: k = 0 ... {K}")
for k in range(K + 1):
    plogits = run_patch(k)
    pprobs  = torch.softmax(plogits, dim=-1)
    pm      = metric(plogits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
    rec     = (pm - corrupt_metric) / (clean_metric - corrupt_metric)
    site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
    sweep_rows.append({
        "k": k, "site": site,
        "p_clean_patched":   pprobs[CLEAN_ANS_ID].item(),
        "p_corrupt_patched": pprobs[CORRUPT_ANS_ID].item(),
        "patched_metric":    pm,
        "recovery":          rec,
    })
df_sweep = pd.DataFrame(sweep_rows)
out_csv = outputs_dir / "nb02_qwen3_8b_patching_sweep.csv"
df_sweep.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.name}")

display(df_sweep.set_index("k").style.set_caption("patching sweep [Qwen3-8B]")
        .format({"recovery": "{:.4f}", "patched_metric": "{:.4f}",
                 "p_clean_patched": "{:.4f}", "p_corrupt_patched": "{:.4f}"}))


In [ ]:
# P(answer) after patch
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["p_clean_patched"],
        label=f"P({CLEAN_ANSWER})  patched",   marker="o", markersize=3)
ax.plot(df_sweep["k"], df_sweep["p_corrupt_patched"],
        label=f"P({CORRUPT_ANSWER}) patched",  marker="s", markersize=3, linestyle="--")
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title(f"Activation Patching — P(answer) after patch  [Qwen3-8B]\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.set_xticks(range(0, K + 1, max(1, K // 9)))
ax.grid(True, alpha=0.3)

cross = df_sweep[df_sweep["p_clean_patched"] > df_sweep["p_corrupt_patched"]]
if len(cross) > 0:
    k_cross = int(cross["k"].min())  # pyright: ignore[reportArgumentType]
    ax.axvline(k_cross, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_cross}: P({CLEAN_ANSWER.strip()}) > P({CORRUPT_ANSWER.strip()})")
ax.legend()
plt.tight_layout()
out = outputs_dir / "nb02_qwen3_8b_patching_probs.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(cross) > 0:
    print(f"最初に確率が入れ替わる layer: k={k_cross}")


In [ ]:
# Recovery curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["recovery"], marker="o", markersize=4, color="steelblue", label="recovery")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("recovery")
ax.set_title(f"Activation Patching — Recovery by Layer  [Qwen3-8B]\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.set_xticks(range(0, K + 1, max(1, K // 9)))
ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3)

transition = df_sweep.loc[df_sweep["recovery"] >= 0.5, "k"]
if len(transition) > 0:
    k_transition = int(transition.min())
    ax.axvline(k_transition, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_transition}: recovery≥0.5")
    ax.legend()
plt.tight_layout()
out = outputs_dir / "nb02_qwen3_8b_recovery_curve.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(transition) > 0:
    print(f"最初に recovery≥0.5 になる layer: k={k_transition}")


## 8. Activation Patching Grid — layer × position

In [ ]:
def run_patch_at_position(k: int, patch_pos: int) -> torch.Tensor:
    patch_vec = clean_hs[k][0, patch_pos, :].to(device)
    if k == 0:
        target_module = model.model.embed_tokens
    elif k < K:
        target_module = model.model.layers[k - 1]
    else:
        target_module = model.model.norm

    def hook(module, inp, out):
        if isinstance(out, tuple):
            first = out[0].clone()
            first[0, patch_pos, :] = patch_vec
            return (first,) + out[1:]
        if isinstance(out, list):
            first = out[0].clone()
            first[0, patch_pos, :] = patch_vec
            return [first] + list(out[1:])
        patched = out.clone()
        patched[0, patch_pos, :] = patch_vec
        return patched

    handle = target_module.register_forward_hook(hook)
    try:
        patched_out = model(
            **corrupt_inputs,
            output_hidden_states=False,
            output_attentions=False,
            use_cache=False,
        )
    finally:
        handle.remove()
    return patched_out.logits[0, corrupt_pos, :].float().detach().cpu()


In [ ]:
# grid を計算
input_ids_clean_grid   = clean_inputs["input_ids"][0].detach().cpu().tolist()
input_ids_corrupt_grid = corrupt_inputs["input_ids"][0].detach().cpu().tolist()
T_grid = len(input_ids_clean_grid)

recovery_grid         = np.zeros((K + 1, T_grid), dtype=np.float64)
text_grid             = [["" for _ in range(T_grid)] for _ in range(K + 1)]
patched_top1_is_clean = np.zeros((K + 1, T_grid), dtype=bool)

_clean_metric_grid   = metric(clean_logits,   CLEAN_ANS_ID, CORRUPT_ANS_ID)
_corrupt_metric_grid = metric(corrupt_logits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
_metric_range        = _clean_metric_grid - _corrupt_metric_grid

patch_rows = []
print(f"Patching grid: (K+1) x T = ({K+1}) x ({T_grid}) = {(K+1)*T_grid} forwards")
print("  k=", end="", flush=True)
for k in range(K + 1):
    for patch_pos in range(T_grid):
        ll_logits  = logit_lens(clean_hs, k, patch_pos).float().detach().cpu()
        ll_top1_id = int(ll_logits.argmax().item())

        plogits  = run_patch_at_position(k, patch_pos)
        pm       = metric(plogits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
        rec      = (pm - _corrupt_metric_grid) / _metric_range
        ptop1_id = int(plogits.argmax().item())
        is_clean = (ptop1_id == CLEAN_ANS_ID)

        recovery_grid[k, patch_pos]         = rec
        text_grid[k][patch_pos]             = short_piece(ll_top1_id)
        patched_top1_is_clean[k, patch_pos] = is_clean

        site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
        patch_rows.append({
            "layer_k":   k,
            "patch_pos": patch_pos,
            "patch_site": site,
            "input_token_clean":   short_piece(input_ids_clean_grid[patch_pos]),
            "input_token_corrupt": short_piece(input_ids_corrupt_grid[patch_pos]),
            "clean_logit_lens_top1_token": short_piece(ll_top1_id),
            "patched_metric": pm,
            "recovery": rec,
            "patched_top1_token": short_piece(ptop1_id),
            "patched_top1_is_clean_answer": bool(is_clean),
        })
    print(f"{k:2d} ", end="", flush=True)
print()

df_patch_grid = pd.DataFrame(patch_rows)
csv_out = outputs_dir / "nb02_qwen3_8b_activation_patching_grid_recovery.csv"
df_patch_grid.to_csv(csv_out, index=False)
print(f"Saved: {csv_out.name}")


In [ ]:
# heatmap として描画
fig_w = max(8.0, 1.6 * T_grid)
fig_h = max(10.0, 0.28 * (K + 1))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
cmap = plt.get_cmap("viridis")
im = ax.imshow(recovery_grid, aspect="auto", vmin=0.0, vmax=1.0, cmap=cmap, origin="lower")
ax.set_title(f"Activation patching grid: clean → corrupt  [Qwen3-8B]\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.set_xlabel("Patched position")
ax.set_ylabel("Layer k")

def _patch_pos_label(pos: int) -> str:
    c = short_piece(input_ids_clean_grid[pos])
    r = short_piece(input_ids_corrupt_grid[pos])
    if c == r:
        return f"{pos}: {c}"
    return f"{pos}: {c}/{r}"

ax.set_xticks(range(T_grid))
ax.set_xticklabels([_patch_pos_label(t) for t in range(T_grid)], rotation=45, ha="right")

ylabels = []
for k in range(K + 1):
    if k == 0:
        ylabels.append("0 emb")
    elif k == K:
        ylabels.append(f"{k} norm")
    else:
        ylabels.append(str(k))
ax.set_yticks(range(K + 1))
ax.set_yticklabels(ylabels)

for k in range(K + 1):
    for pos in range(T_grid):
        rec_clipped = float(max(0.0, min(1.0, recovery_grid[k, pos])))
        rgba        = cmap(rec_clipped)
        lum         = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
        text_color  = "white" if lum < 0.5 else "black"
        ax.text(pos, k, text_grid[k][pos],
                ha="center", va="center", fontsize=7, color=text_color)
        if patched_top1_is_clean[k, pos]:
            ax.add_patch(Rectangle((pos - 0.5, k - 0.5), 1, 1,
                                   fill=False, edgecolor="magenta", linewidth=2.2))

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("recovery at final position")
fig.tight_layout()
png_out = outputs_dir / "nb02_qwen3_8b_activation_patching_grid_recovery.png"
fig.savefig(png_out, dpi=200)
plt.show()
print(f"Saved: {png_out.name}")


## 9. まとめと 4B / 1.7B との比較

### 3 モデル比較表

| 観点 | Qwen3-1.7B | Qwen3-4B | **Qwen3-8B** |
|------|------------|----------|--------------|
| layer 数 K | 28 | 36 | **36** |
| hidden_size | 2048 | 2560 | **4096** |
| num_attention_heads | 16 | 32 | **32** |
| num_key_value_heads | 8 | 8 | **8** |
| intermediate_size | 6144 | 9728 | **12288** |
| tie_word_embeddings | True | True | **False** |
| answer token ID | (同一) | ` Tokyo`=26194 / ` Paris`=12095 | (同一) |
| 最終層 clean P(` Tokyo`) | 0.4169 | 0.8946 | **0.4687** |
| 最終層 corrupt P(` Paris`) | 0.5246 | 0.6346 | **0.5345** |
| `clean_metric` (logit diff) | 7.4766 | 11.6953 | **8.1406** |
| `corrupt_metric` | −8.3281 | −11.9844 | **−7.6406** |
| 最初に recovery ≥ 0.5 となる k | k = 21 | k = 25 | **k = 27** |
| 上記境界 layer の比率 k/K | 21/28 = 0.75 | 25/36 ≈ 0.69 | **27/36 = 0.75** |
| 上記境界以降に残る layer 数 (K − k) | 7 | 11 | **9** |

### 観察と解釈

- **境界 layer は深くなる方向にずれた**：
  4B → 8B では K は同じ 36 のままだが、境界 layer は k=25 → **k=27** と深くなった。
  比率 k/K で見ても 0.69 → 0.75 で深い側にシフト。
  「同じ深さでも、より広いモデルでは Tokyo を最終的に書き込む処理が後ろにずれる」という結果。
  処理が完了するまでの末尾層数（K − k）は 11 → 9 となり、わずかに圧縮されている。

- **Patching Grid の補完構造は維持されたが、より緩やか**：
  - **pos=3（` Japan` / ` France`）に patch**：k=0〜24 では recovery > 0.85（一部 > 1.0 の overshoot）。
    k=25 から急減し、k=27 以降は recovery < 0.4。
  - **pos=4（最後の ` is`）に patch**：k=22〜24 では recovery が**負**（−0.07 〜 −0.09）、
    k=25 から正に転じ、k=27 で 0.6、k=33 で 1.0 に到達。
  - 1.7B / 4B では境界が比較的シャープだったのに対し、8B は **k=24〜32 の幅広い遷移帯**が
    存在する。これは「Tokyo を最終 position に書き込む処理が、複数の連続する layer に
    分散している」と読める。capacity の増加によって、特定の計算が複数 layer に
    分割されるパターンかもしれない。
  - また pos=4 の k=22〜24 で **recovery が負**になるのも特徴的：
    embedding 空間と整合的でない hidden state を注入することで、
    むしろ Paris 側へ確率が押し戻される（一時的な「妨害」効果）。

- **中間層の latent language が 4B / 1.7B より顕著**：
  8B の grid セル内文字（clean activation の logit lens top-1）には、中国語 token が
  **多数・幅広い layer**に現れる：
  - **`这座城市`**（this city）at (10, 2)
  - **`城市的`**（city's）at 複数の (k, pos)
  - **`与中国`**（with China）at (12, 3), (13, 3)
  - **`中国的`**（China's）at (24, 2), (25, 2)
  - **`在日本`**（in Japan）at (9, 3), (10, 3), (11, 3), (15, 3), (16, 3)
  - **`和地区`**（and region）at (12, 4)
  - **`之城`**（city of）at (14, 1), (15, 1), (16, 1)
  - その他 `坐落于` `首富` `范冰` 等
  - 加えて k=0 には ロシア語片（`Почем` `Дмитр`）も観察される

  これは Wendler et al. (2024) / Zhong et al. (2025) が示した
  「**latent language** が中間層に立ち上がる」現象の、Qwen3 系列における鮮明な事例。
  1.7B では `坐落于` 1 トークン程度しか観察されなかったのに対し、8B は中国語の
  「都市・場所・国」関連の概念がはるかに広く中間層に分散している。
  capacity が増えるほど latent representation が豊富になり、ここでは特に中国語の都市・地理
  関連 token が顕著に立ち上がる。Qwen3 が中国の Alibaba 製で**事前学習データに中国語が
  豊富に含まれている**ことと整合的な観察。

- **確率の絶対値は意外と高くない**：
  パラメータが増えれば P(` Tokyo`) も上がるという素朴な予想は **外れた**。
  8B の P(` Tokyo`) は 0.47 で、4B の 0.89 より大幅に低い（1.7B の 0.42 とほぼ同水準）。
  原因として `tie_word_embeddings = False` の影響が考えられる：
  - 4B は embed と lm_head が同じ重み → 出力分布が個別の単語にピークしやすい
  - 8B は両者が独立 → 出力分布がより滑らか・分散的になる
  - 8B の top-10 に ` Beijing` が入る点も、「アジアの首都」という近傍に確率質量が
    広く割れていることの傍証

### 講義デモへの示唆

- 1.7B / 4B / 8B を並べて見ると、**Qwen3 系列の共通構造**（境界 layer で補完的に切り替わる
  patching の挙動、latent language の出現）と、**サイズ・実装の違いによる差**
  （境界 layer の位置、確信度の絶対値、tie embedding の有無）が両方観察できる。
- 特に **「大きいモデルが必ずしも『鋭い分布』を持つわけではない」**
  という観察は、`tie_word_embeddings` の設計選択を含めた **「モデルカード／config を読む」
  実践**の重要性を講義で強調する素材になる。
- 8B の grid に現れる中国語 latent language の豊富さは、**多言語事前学習の効果**を
  内部表現として可視化する強い例示。同じ実験フレームを使って異なるモデル系列
  （例：英語中心の LLM）と比較すれば、より一般的な議論につながる。
